In [73]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

The third line ensures that if we later add more columns, none are hidden.

The fourth makes numerical outputs easier to read.

## 0. Load the data set

In [77]:
water = pd.read_csv("../data/raw/water_potability.csv")

## 1. Initial Inspection

The purpose of this section is to gain a first impression of the dataset before performing any quality assessment or cleaning.

In [79]:
water.sample(5, random_state=42)

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
2947,NaN,183.52,20461.25,7.33,333.12,356.37,20.18,67.02,4.89,0
2782,6.64,188.91,32873.82,6.79,333.85,336.56,14.71,67.84,4.56,1
1644,7.85,224.06,23264.11,5.92,300.40,387.97,13.41,43.08,2.49,0
70,7.16,183.09,6743.35,3.80,277.60,428.04,9.80,90.04,3.88,0
2045,6.62,179.24,26392.86,9.31,NaN,496.36,12.79,78.26,4.45,1


## 2. Dataset Dimensions


In [48]:
water.shape

(3276, 10)

The dataset contains 3276 observations and 10 variables. Based on its size, the dataset is suitable for exploratory analysis and can be processed comfortably on a standard personal computer

## 3. Dataset Structure and Data Types

This section examines the column names, data types, non-null counts, and memory usage of the dataset.

In [49]:
water.info()

<class 'pandas.DataFrame'>
RangeIndex: 3276 entries, 0 to 3275
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ph               2785 non-null   float64
 1   Hardness         3276 non-null   float64
 2   Solids           3276 non-null   float64
 3   Chloramines      3276 non-null   float64
 4   Sulfate          2495 non-null   float64
 5   Conductivity     3276 non-null   float64
 6   Organic_carbon   3276 non-null   float64
 7   Trihalomethanes  3114 non-null   float64
 8   Turbidity        3276 non-null   float64
 9   Potability       3276 non-null   int64  
dtypes: float64(9), int64(1)
memory usage: 256.1 KB


The dataset contains nine continuous numerical variables and one binary target variable, Potability. The target is stored as an integer, where 0 represents non-potable water and 1 represents potable water. The ph, Sulfate, and Trihalomethanes columns contain missing values, while the remaining variables are complete. No obvious data type conversion is required at this stage.

## 4. Summary Statistics

In [ ]:
water.describe().T

,count,mean,std,min,25%,50%,75%,max
ph,2785.00,7.08,1.59,0.00,6.09,7.04,8.06,14.00
Hardness,3276.00,196.37,32.88,47.43,176.85,196.97,216.67,323.12
Solids,3276.00,22014.09,8768.57,320.94,15666.69,20927.83,27332.76,61227.20
Chloramines,3276.00,7.12,1.58,0.35,6.13,7.13,8.11,13.13
Sulfate,2495.00,333.78,41.42,129.00,307.70,333.07,359.95,481.03
Conductivity,3276.00,426.21,80.82,181.48,365.73,421.88,481.79,753.34
Organic_carbon,3276.00,14.28,3.31,2.20,12.07,14.22,16.56,28.30
Trihalomethanes,3114.00,66.40,16.18,0.74,55.84,66.62,77.34,124.00
Turbidity,3276.00,3.97,0.78,1.45,3.44,3.96,4.50,6.74
Potability,3276.00,0.39,0.49,0.00,0.00,0.00,1.00,1.00


The variables operate on substantially different numerical scales. For example, Solids has values ranging from approximately 321 to 61,227, while Turbidity ranges from approximately 1.45 to 6.74. This scale difference may become important if predictive modelling is introduced later.

The mean and median values for several variables are relatively close, although this alone is not enough to determine their distribution shapes. Visual analysis will be required during exploratory analysis.

The ph variable ranges from 0 to 14, covering the full conventional pH scale. Values at the extremes may require closer investigation because they could represent unusual water conditions or potential outliers.

The Potability mean of approximately 0.39 indicates that about 39% of the observations are classified as potable.

## 5. Missing Values Assessment

In [81]:
missing = (
  water.isna().sum().to_frame("Missing Values")
)

missing["Percentage"] = (
    missing["Missing Values"]
    / len(water)
) * 100

missing.sort_values("Percentage", ascending=False)


,Missing Values,Percentage
Sulfate,781,23.84
ph,491,14.99
Trihalomethanes,162,4.95
Hardness,0,0.00
Chloramines,0,0.00
Solids,0,0.00
Conductivity,0,0.00
Organic_carbon,0,0.00
Turbidity,0,0.00
Potability,0,0.00


Missing values are concentrated in three variables. Sulfate has the highest missingness at 23.84%, followed by ph at 14.99% and Trihalomethanes at 4.95%. Because Sulfate and ph have substantial missingness, immediately deleting all affected rows could remove a meaningful portion of the dataset. The missing-data patterns and suitable treatment methods should therefore be examined during the data-cleaning sprint.

In [82]:
rows_with_missing = water.isna().any(axis=1).sum()
missing_row_percentage = rows_with_missing / len(water) * 100

print(f"Rows with at least one missing value: {rows_with_missing}")
print(f"Percentage of affected rows: {missing_row_percentage:.2f}%")

Rows with at least one missing value: 1265
Percentage of affected rows: 38.61%


Missing values occur in Sulfate, ph, and Trihalomethanes. Overall, 1,265 rows, representing 38.61% of the dataset, contain at least one missing value. This makes complete-case deletion potentially costly and supports a more careful missing-data treatment during the cleaning stage.

## 6. Duplicate Assessment

In [58]:
water.duplicated().sum()

np.int64(0)

No exact duplicate records were identified. Therefore, no rows need to be removed on the basis of complete duplication. This does not yet rule out possible logical duplicates or unusually similar observations.

## 7. Target Variable

In [70]:
potability_counts = (
    water["Potability"]
    .value_counts()
    .rename(index={0: "Not Potable", 1: "Potable"})
    .to_frame("Count")
)

potability_counts["Percentage"] = (
    potability_counts["Count"] / len(water) * 100
)

potability_counts

,Count,Percentage
Potability,,
Not Potable,1998,60.99
Potable,1278,39.01


The target variable shows a mild to moderate class imbalance, with approximately 61% of samples classified as non-potable and 39% as potable.

## 8. Data Understanding Summary

The dataset contains 3,276 water samples and 10 variables, including nine water-quality measurements and one binary potability target. All variables are numerical, and no exact duplicate records were found.

Missing data occurs in Sulfate, ph, and Trihalomethanes, with Sulfate having the largest proportion of missing values. These values require a carefully justified treatment during the cleaning stage.

The target variable is moderately imbalanced, with non-potable samples accounting for approximately 61% of the dataset and potable samples accounting for approximately 39%.

The summary statistics also indicate that the features operate on very different scales and that some variables contain extreme minimum or maximum values that should be investigated visually before any outlier treatment is considered.

Based on this assessment, the next sprint will focus on evaluating missing-data patterns, checking the validity of numerical ranges, standardising column names where necessary, and preparing a cleaned dataset for exploratory analysis.